# Fig. 3(a) de Brod & Combes com um oscilador em cada átomo (estados de Fock)

Este notebook explica `calcular_fock_N.py`, `calcular_fock_N_lenta.py` (regime de mola lenta, Seção 14) e `plot_fock_N.py`. Para cada variável e cada expressão ele diz o que ela representa, mostra a fórmula e indica **qual função do código a implementa**. As células de código importam `calcular_fock_N.py` e fazem contas curtas. Elas **não** sobrescrevem os `.npz` nem as figuras.

**Referências usadas**
* `daniel2016.pdf`: Brod & Combes, *A passive CPHASE gate via cross-Kerr nonlinearities*. A Fig. 3(a) mostra a fidelidade média $F$ em função da largura $\sigma$ do pacote para $N=1,2,3,4,5,10,20$ sítios contrapropagantes ($\chi=1000$, $\gamma=1$, $\omega_0=0$).
* `matriz_S_dois_sitios_osciladores.pdf`: matriz $S$ de dois fótons para dois sítios, cada átomo com um oscilador (Eqs. E1–E12).
* `matriz_S_N_sitios_osciladores.pdf`: generalização para $N$ sítios (Eqs. N1–N12), demonstrada por indução.
* `2foton-1sitio-fock/`: o caso de um sítio, com a mesma definição de fidelidade para estado misto.
* `infidelidade/`: a Fig. 3 original (sem osciladores), de onde vêm as cores.

**O sistema.** Há $N$ sítios de interação. No sítio $j$ estão dois átomos, $A^{(j)}$ e $B^{(j)}$, ligados por um cross-Kerr $\chi_j$. O fóton $a$ percorre a cadeia $1\to N$ e interage só com os átomos $A^{(j)}$. O fóton $b$ percorre $N\to1$ e interage só com os $B^{(j)}$ (contrapropagação). **Cada átomo tem o seu oscilador harmônico** ("mola"): $(ja)$ no átomo $A^{(j)}$ e $(jb)$ no $B^{(j)}$. São $2N$ osciladores independentes, cada um começando num estado de Fock $|n\rangle$.

**O que muda em relação a Brod–Combes.** Ao atravessar um sítio, o fóton pode trocar quanta com o oscilador daquele átomo, e a sua cor muda de $\pm\omega^{\rm osc}$. Os osciladores terminam num estado $|m\rangle$ que não é medido. Por isso o estado dos fótons na saída é **misto**, e a fidelidade tem de ser a de um estado misto com um estado puro ([Wikipedia, *Fidelity of quantum states*](https://en.wikipedia.org/wiki/Fidelity_of_quantum_states)), como em `2foton-1sitio-fock`.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import calcular_fock_N as c

## 1. Parâmetros (topo de `calcular_fock_N.py`)

Todas as frequências estão em unidades de $\gamma$. Os parâmetros dos sítios são os da Fig. 3 do artigo. Assim, com os osciladores desligados ($S=0$), a Fig. 3(a) original é reproduzida.

**Sítios**

| variável | símbolo | significado |
|---|---|---|
| `chi` | $\chi_j$ | acoplamento cross-Kerr entre $A^{(j)}$ e $B^{(j)}$ (`np.inf`: limite saturado) |
| `gam` | $\gamma_j$ | taxa de emissão de cada átomo no guia (largura de linha) |
| `Delta` | $\Delta_j$ | dessintonia dos átomos |
| `omega0` | $\omega_0$ | dessintonia do centro do pacote em relação a $\Delta$ |
| `omega_c` | $\omega_c=\Delta+\omega_0$ | frequência central do pacote (derivada) |
| `phi` | $\phi$ | fase da porta CPHASE ideal ($\pi$) |
| `N_values` | $N$ | números de sítios (as curvas da Fig. 3(a)) |

**Osciladores.** Cada entrada de `casos` é `(w_a, S_a, n_a, w_b, S_b, n_b)`, igual em todos os sítios (para sítios diferentes, edite `montar_cadeia`):

| variável | símbolo | significado |
|---|---|---|
| `w_a`, `w_b` | $\omega^{(ja)},\ \omega^{(jb)}$ | frequência do oscilador do átomo $A^{(j)}$ / $B^{(j)}$ |
| `S_a`, `S_b` | $S=(2g/\omega)^2=\beta^2$ | fator de Huang–Rhys ($g$ é o acoplamento átomo–oscilador) |
| `n_a`, `n_b` | $n^{(j)}_a,\ n^{(j)}_b$ | estado de Fock inicial |

`S = 0` desliga os osciladores e dá a curva de Brod–Combes.

**Fidelidade e truncamento**

| variável | significado |
|---|---|
| `referencia` | como se define o fóton ideal de saída (Seção 6) |
| `max_excitados` | número máximo de osciladores que terminam com $m\ne n$ na soma sobre $m$ (Seção 9) |
| `dq_max` | $\lvert m-n\rvert$ máximo de cada oscilador |
| `tol_sobreposicao` | descarta configurações cuja sobreposição gaussiana com o fóton ideal é menor que isso |

**Numéricos**: `sigma_min`, `sigma_max`, `n_sigma` (varredura log em $\sigma$); `k_sigma` (meia-janela de frequências em unidades de $\sigma$); `pts_per_width`, `n_min`, `n_max` (grade); `eps_fc`, `eps_peso` (cortes de Franck–Condon); `n_proc` (processos em paralelo).

In [ ]:
print('chi =', c.chi, ' gamma =', c.gam, ' Delta =', c.Delta, ' omega0 =', c.omega0, ' phi =', c.phi)
print('N_values =', c.N_values)
print('casos =', c.casos)
print('max_excitados =', c.max_excitados, ' dq_max =', c.dq_max, ' referencia =', c.referencia)

## 2. O oscilador: fatores de Franck–Condon — `FC`, `oscilador`, `niveis`

O átomo excitado desloca o oscilador. Os estados vestidos $|k\rangle$ (átomo excitado com o oscilador deslocado) se sobrepõem aos estados de Fock $|n\rangle$ pelos fatores de Franck–Condon
$$F_{kn}=\langle k|D(\beta)|n\rangle,\qquad \beta=\frac{2g}{\omega}=\sqrt S .$$
Para $k\ge n$,
$$F_{kn}=\sqrt{\frac{n!}{k!}}\;\beta^{k-n}\,e^{-\beta^2/2}\,L_n^{(k-n)}(\beta^2),$$
e $F_{kn}=(-1)^{n-k}F_{nk}$ para $k<n$ (**`FC`**).

**`oscilador(w, S, n)`** devolve um dicionário:

| chave | significado |
|---|---|
| `w`, `S`, `n`, `g` | $\omega$, $S$, Fock inicial $n$, $g=\sqrt S\,\omega/2$ |
| `desloc` | $4g^2/\omega=S\omega$ (deslocamento polarônico da ressonância) |
| `F`, `K` | matriz $F_{km}$ ($0\le k,m<K$). O corte $K$ cresce até os níveis úteis não encostarem nele (`eps_fc`) |
| `finais` | estados finais $m\ne n$ permitidos ($\lvert m-n\rvert\le$ `dq_max`) |
| `niv[m]` | os níveis $k$ com $\lvert F_{km}F_{kn}\rvert>$ `eps_peso` e esses produtos |

**`niveis(osc, m)`** devolve os $k$ que entram nas somas para a transição $n\to m$.

In [ ]:
o = c.oscilador(1.0, 0.01, 1)
print('K =', o['K'], '  finais =', o['finais'])
print('niveis k para m=n:', c.niveis(o, 1), '  pesos F_kn^2 =', np.round(o['niv'][1][1], 6))
print('sum_k F_kn^2 =', np.sum(o['F'][:, 1]**2), '(D(beta) e unitario)')

## 3. Um fóton num sítio — `Gam`, `t_mn`, `chiK`

Para o oscilador $(jc)$, com $c=a,b$ [Eq. (E2)/(N2)]:
$$\Gamma^{(jc)}_{kn}(x)=\frac{\gamma_j}{2}+i\Big(\Delta_j-\frac{4(g^{(jc)})^2}{\omega^{(jc)}}+(k-n)\,\omega^{(jc)}-x\Big)\qquad(\texttt{Gam})$$

$$\langle y,m|x,n\rangle_{jc}=t^{(jc)}_{mn}(x)\,\delta\big(y-x+(m-n)\omega^{(jc)}\big),\qquad
t^{(jc)}_{mn}(x)=\delta_{mn}-\gamma_j\sum_k\frac{F^{(jc)}_{km}F^{(jc)}_{kn}}{\Gamma^{(jc)}_{kn}(x)}\qquad(\texttt{t\_mn})$$

O fóton que entra com cor $x$ e leva o oscilador de $|n\rangle$ a $|m\rangle$ sai com $y=x-(m-n)\omega$: a energia que ele perde fica no oscilador.

**Sem oscilador** ($S=0$, $F_{kn}=\delta_{kn}$): $t_{nn}=1-\gamma/\Gamma=-\Gamma^*/\Gamma$, a fase pura de Brod–Combes.

**Kernel de Kerr.** No termo de dois fótons aparece $\big[1+2i\chi_j/\Sigma\big]^{-1}$, com
$$\Sigma=\Gamma^{(ja)}_{k_an_a}(x_a)+\Gamma^{(jb)}_{k_bn_b}(x_b)=\gamma_j+i\Big(2\Delta_j-S_a\omega_a-S_b\omega_b+(k_a-n_a)\omega_a+(k_b-n_b)\omega_b-E\Big),$$
que só depende de $E=x_a+x_b$. A função **`chiK`** devolve $\chi_j\,[1+2i\chi_j/\Sigma]^{-1}=\chi_j\Sigma/(\Sigma+2i\chi_j)$, ou $\Sigma/(2i)$ se $\chi=\infty$.

Verificação: um fóton conserva a probabilidade, $\sum_m\int d\nu\,|\xi(\nu)|^2|t_{mn}(\nu)|^2=1$.

In [ ]:
sit = c.montar_cadeia(1, 1.0, 0.01, 1, 1.0, 0.01, 1)[0]
nu = np.linspace(-12, 12, 24001); h = nu[1] - nu[0]
xi2 = c.xi_in(nu, 1.0)**2
print('norma de 1 foton =', sum(np.sum(xi2 * np.abs(c.t_mn(sit, sit['a'], m, nu))**2) * h
                                for m in [1] + sit['a']['finais']))

plt.figure(figsize=(6, 3))
x = np.linspace(-4, 4, 800)
plt.plot(x, np.abs(c.t_mn(sit, sit['a'], 1, x))**2, label=r'$|t_{nn}|^2$ (elástico)')
for m in sit['a']['finais']:
    plt.plot(x, np.abs(c.t_mn(sit, sit['a'], m, x))**2, label=fr'$|t_{{{m}1}}|^2$')
plt.xlabel(r'$x-\Delta$ [$\gamma$]'); plt.legend(); plt.title(r'$S=0.01,\ \omega=1,\ n=1$'); plt.tight_layout(); plt.show()

A figura mostra o que vai importar. Na ressonância, cerca de 3% de um fóton sai **inelástico** em cada sítio, mesmo com $S=0.01$: o átomo ressonante interage fortemente com o fóton, e o oscilador "pega carona" nessa interação. Esse efeito se acumula ao longo dos $N$ sítios.

## 4. As cores do fóton ao longo da cadeia — `deslocamentos`

Uma **configuração** dos osciladores é o par de tuplas $(m_a, m_b)$, com o estado final de cada um dos $2N$ osciladores. Os deslocamentos de cor são [Eq. (N1)]
$$\delta^{(k)}_a=\big(m^{(k)}_a-n^{(k)}_a\big)\,\omega^{(ka)},\qquad
\delta^{(k)}_b=\big(m^{(k)}_b-n^{(k)}_b\big)\,\omega^{(kb)},\qquad D_k=\delta^{(k)}_a+\delta^{(k)}_b .$$

O fóton $a$ vai de $1$ a $N$ e o fóton $b$ de $N$ a $1$. Fora da interação, as cores são [Eqs. (N5)–(N8)]
$$x^{(k)}_a=\nu_a-\sum_{l<k}\delta^{(l)}_a,\qquad y^{(k)}_a=\omega_a+\sum_{l>k}\delta^{(l)}_a,\qquad
x^{(k)}_b=\nu_b-\sum_{l>k}\delta^{(l)}_b,\qquad y^{(k)}_b=\omega_b+\sum_{l<k}\delta^{(l)}_b .$$
* $x^{(k)}$ é a cor de **entrada** no sítio $k$: a cor inicial menos o que o fóton já deixou nos osciladores do caminho.
* $y^{(k)}$ é a cor de **saída** do sítio $k$: a cor final mais o que ele ainda vai deixar.

No código (**`deslocamentos`**), `pa[k]` $=\sum_{l<k}\delta^{(l)}_a$ (o que o fóton $a$ deixou antes do sítio $k$) e `sb[k]` $=\sum_{l>k}\delta^{(l)}_b$ (o que o fóton $b$ deixou antes do sítio $k$).

In [ ]:
cad = c.montar_cadeia(4, 1.0, 0.01, 1, 1.0, 0.01, 1)
da, db, pa, sb = c.deslocamentos(cad, (2, 1, 0, 1), (1, 1, 1, 0))
print('delta_a =', da, '  pa =', pa)
print('delta_b =', db, '  sb =', sb)

## 5. A matriz $S$ de $N$ sítios [Eqs. (N9)–(N11)]

$$\langle\omega_a\omega_b;m|S^{(N)}|\nu_a\nu_b;n\rangle=\underbrace{F^{(N)}}_{\text{nenhuma interação}}+\sum_{j=1}^N\underbrace{X^{(N)}_j}_{\text{interação só no sítio }j}.$$

**Termo livre** [Eq. (N10)]: cada fóton atravessa os $N$ sítios sozinho, e a amplitude é o produto das amplitudes de um fóton, com as cores deslocadas:
$$F^{(N)}=\Big[\prod_{k=1}^N t^{(ka)}_{m^{(k)}_an^{(k)}_a}\big(x^{(k)}_a\big)\Big]\Big[\prod_{k=1}^N t^{(kb)}_{m^{(k)}_bn^{(k)}_b}\big(x^{(k)}_b\big)\Big]\,
\delta\Big(\omega_a-\nu_a+\sum_k\delta^{(k)}_a\Big)\,\delta\Big(\omega_b-\nu_b+\sum_k\delta^{(k)}_b\Big).$$

**Termo de interação no sítio $j$** [Eq. (N11)]:
$$X^{(N)}_j=-i\frac{\chi_j\gamma_j^2}{\pi}\,\delta\Big(\omega_a+\omega_b+\sum_kD_k-\nu_a-\nu_b\Big)
\underbrace{\prod_{k<j}t^{(ka)}\big(x^{(k)}_a\big)}_{a\text{ antes}}\underbrace{\prod_{k>j}t^{(ka)}\big(y^{(k-1)}_a\big)}_{a\text{ depois}}
\underbrace{\prod_{k>j}t^{(kb)}\big(x^{(k)}_b\big)}_{b\text{ antes}}\underbrace{\prod_{k<j}t^{(kb)}\big(y^{(k+1)}_b\big)}_{b\text{ depois}}
\;C_j\big(x^{(j)}_a,x^{(j)}_b;y^{(j)}_a,y^{(j)}_b\big),$$
com o núcleo de Kerr [Eq. (N4)]
$$C_j(x_a,x_b;y_a,y_b)=\sum_{k_a,k_b}\frac{F^{(ja)}_{k_am_a}F^{(ja)}_{k_an_a}F^{(jb)}_{k_bm_b}F^{(jb)}_{k_bn_b}}
{\Gamma^{(ja)}_{k_an_a}(x_a)\,\Gamma^{(jb)}_{k_bn_b}(x_b)\,\Gamma^{(ja)}_{k_am_a}(y_a)\,\Gamma^{(jb)}_{k_bm_b}(y_b)}\;
\frac{1}{1+\dfrac{2i\chi_j}{\Gamma^{(ja)}_{k_an_a}(x_a)+\Gamma^{(jb)}_{k_bn_b}(x_b)}} .$$

**Leitura.** O fóton $a$ atravessa sozinho os sítios $1,\dots,j-1$ e o $b$ atravessa sozinho $N,\dots,j+1$. Os dois se encontram no sítio $j$ (núcleo $C_j$). Depois, $a$ segue sozinho por $j+1,\dots,N$ e $b$ por $j-1,\dots,1$. **Não há termos com dois $\chi$**: fótons contrapropagantes se encontram uma única vez. Isso é o termo (IV) que se anula nos PDFs, por um argumento de polos no semiplano inferior. A delta só conserva a energia **total**, incluindo os $2N$ osciladores.

Sem osciladores, $t\to-\Gamma^*/\Gamma$ e tudo recai na fórmula de $N$ sítios de Brod–Combes, $\sum_j A^{N-j}B^{j-1}$.

## 6. O pacote e o fóton ideal — `xi_in`, `fase_ref`, `xi_ideal_conj`

**Pacote de entrada** (Eq. (4) do artigo, **`xi_in`**):
$$\xi(\omega)=\frac{1}{(2\pi\sigma^2)^{1/4}}\exp\!\Big[-\frac{(\omega-\omega_c)^2}{4\sigma^2}\Big].$$

**Remoção da deformação de um fóton.** Brod & Combes comparam a saída com o fóton deformado pelo processo linear, $\xi\to(-\Gamma^*/\Gamma)^N\xi$, uma fase pura. Com os osciladores, a parte elástica da cadeia, $\prod_kt^{(kc)}_{nn}$, não tem mais módulo 1. O fóton ideal de saída é então
$$\tilde\xi_c(\omega)=u_c(\omega)\,\xi(\omega),\qquad |u_c|=1,$$
com duas opções (variável `referencia`, função **`fase_ref`**):
* `'elastica'` (padrão): $u_c=\prod_kt^{(kc)}_{nn}\big/\big|\prod_kt^{(kc)}_{nn}\big|$, a fase da amplitude elástica da cadeia. É a mesma escolha (`'media'`) do código de um sítio, e recai em $(-\Gamma^*/\Gamma)^N$ quando $S=0$.
* `'sem_mola'`: $u_c=(-\Gamma^*/\Gamma)^N$ com $\Gamma=\gamma/2+i(\Delta-\omega)$, como se não houvesse osciladores.

**`xi_ideal_conj`** devolve $\tilde\xi_c^*(\omega)$.

## 7. Amplitudes de um fóton projetadas no fóton ideal — `tau_foton`

Projetando o termo livre sobre $\tilde\xi_a$ e integrando contra $\xi(\nu_a)$, a delta elimina $\omega_a=\nu_a-D_a$, com $D_a=\sum_k\delta^{(k)}_a$. Assim
$$\tau_a(m_a)=\int d\nu\;\tilde\xi_a^*(\nu-D_a)\,\xi(\nu)\prod_{k=1}^Nt^{(ka)}_{m^{(k)}_an^{(k)}_a}\Big(\nu-\sum_{l<k}\delta^{(l)}_a\Big),$$
$$\tau_b(m_b)=\int d\nu\;\tilde\xi_b^*(\nu-D_b)\,\xi(\nu)\prod_{k=1}^Nt^{(kb)}_{m^{(k)}_bn^{(k)}_b}\Big(\nu-\sum_{l>k}\delta^{(l)}_b\Big).$$

$\tau_a(m_a)$ é a amplitude de o fóton $a$, sozinho, sair igual ao fóton ideal **e** deixar os seus osciladores em $m_a$. Se $D_a\ne0$, o fóton sai com a cor deslocada, e a sobreposição gaussiana cai como $e^{-D_a^2/8\sigma^2}$. Se ela fica abaixo de `tol_sobreposicao`, o código devolve zero sem integrar.

O termo livre de dois fótons projetado é o produto $\tau_a(m_a)\,\tau_b(m_b)$.

In [ ]:
cad = c.montar_cadeia(5, 1.0, 0.01, 1, 1.0, 0.01, 1)
s = 0.3
h, n = c.grade(s, 5)
print('elastico          tau_a =', c.tau_foton(cad, 'a', (1, 1, 1, 1, 1), s, h, n))
print('ida e volta (D=0) tau_a =', c.tau_foton(cad, 'a', (2, 1, 0, 1, 1), s, h, n))
print('D_a = +1          tau_a =', c.tau_foton(cad, 'a', (2, 1, 1, 1, 1), s, h, n))

## 8. Termo de interação projetado — `termo_interacao`, `termo_nao_linear`

Projetando $X^{(N)}_j$ sobre $\tilde\xi_a\tilde\xi_b$ e integrando contra $\xi(\nu_a)\xi(\nu_b)$, sobra uma integral tripla. Ela é organizada nas cores **no sítio $j$**:
$$u=x^{(j)}_a,\quad v=x^{(j)}_b\ (\text{entradas}),\qquad y=y^{(j)}_a,\quad z=y^{(j)}_b\ (\text{saídas}).$$
Com elas, a delta de energia total vira $y+z=u+v-D_j$ (o sítio $j$ só "vê" o próprio $D_j$), e
$$N_j(m)=-\frac{i\gamma_j^2}{\pi}\sum_{k_a,k_b}F^{(ja)}_{k_am_a}F^{(ja)}_{k_an_a}F^{(jb)}_{k_bm_b}F^{(jb)}_{k_bn_b}
\int dE\;\chi_jK_{k_ak_b}(E)\,P(E)\,Q(E),$$
$$P(E)=\int du\;\alpha(u)\,\beta(E-u),\qquad Q(E)=\int dy\;\alpha'(y)\,\beta'(E-D_j-y),$$
com as quatro funções de uma variável só ($p_a$ = `pa`, $s_b$ = `sb`; $e_a=\sum_{l>j}\delta^{(l)}_a$, $e_b=\sum_{l<j}\delta^{(l)}_b$):

| função | expressão | o que é |
|---|---|---|
| $\alpha(u)$ | $\xi(u+p_a[j])\,\prod_{k<j}t^{(ka)}\big(u+p_a[j]-p_a[k]\big)\,/\,\Gamma^{(ja)}_{k_an_a}(u)$ | $a$ antes |
| $\beta(v)$ | $\xi(v+s_b[j])\,\prod_{k>j}t^{(kb)}\big(v+s_b[j]-s_b[k]\big)\,/\,\Gamma^{(jb)}_{k_bn_b}(v)$ | $b$ antes |
| $\alpha'(y)$ | $\tilde\xi_a^*(y-e_a)\,\prod_{k>j}t^{(ka)}\big(y-\sum_{j<l<k}\delta^{(l)}_a\big)\,/\,\Gamma^{(ja)}_{k_am_a}(y)$ | $a$ depois |
| $\beta'(z)$ | $\tilde\xi_b^*(z-e_b)\,\prod_{k<j}t^{(kb)}\big(z-\sum_{k<l<j}\delta^{(l)}_b\big)\,/\,\Gamma^{(jb)}_{k_bm_b}(z)$ | $b$ depois |

As cores finais são $\omega_a=y-e_a$ e $\omega_b=z-e_b$. O kernel $\chi_jK$ só depende de $E=u+v$, e por isso a integral tripla vira **uma integral em $E$ do produto de duas convoluções**. Essa é a mesma redução do código de um sítio e do de $N$ sítios sem molas. As convoluções são feitas por FFT.

**Grades.** Cada função é amostrada com passo $h$ na rede $\omega_c+h\mathbb Z$, transladada para o centro do seu suporte (**`pontos`**). Assim as convoluções de funções com centros diferentes (cores deslocadas) se alinham exatamente, sem arredondar os deslocamentos. Os fatores $t$ e $\tilde\xi^*$ que se repetem entre configurações ficam em cache (**`t_rede`**, **`ref_rede`**). Só entram os pares $(k_a,k_b)$ com peso de Franck–Condon acima de `eps_peso`.

**`termo_nao_linear`** soma $\mathcal N(m)=\sum_jN_j(m)$. Se a sobreposição gaussiana da energia total, $e^{-(D_a+D_b)^2/16\sigma^2}$, é menor que `tol_sobreposicao`, a soma é zero.

**Verificação.** Abaixo, $\mathcal N(m)$ por FFT é comparado com a integral 3D direta da Eq. (N11) (**`forca_bruta_N`**, sem nenhuma redução), para $N=3$ com sítios diferentes entre si e configurações com deslocamentos.

In [ ]:
cad = c.montar_cadeia(3, 0.3, 0.2, 1, 0.4, 0.15, 1)
cad[1]['gam'], cad[2]['Delta'], cad[0]['chi'] = 1.3, 0.1, 2.0     # sitios diferentes de proposito
s = 0.4
h, n = c.grade(s, 3)
for ma, mb in [((1, 1, 1), (1, 1, 1)), ((2, 1, 0), (1, 1, 1)), ((1, 0, 1), (2, 1, 0))]:
    print(ma, mb, ' FFT:', np.round(c.termo_nao_linear(cad, ma, mb, s, h, n), 6),
          '  direto:', np.round(c.forca_bruta_N(cad, ma, mb, s), 6))

## 9. Fidelidade de estado misto — `fidelidade`, `W1`, `configuracoes`

Entrada lógica: $|\psi\rangle=\sum_kc_k|k\rangle$, com $k\in\{00,01,10,11\}$ (ausência/presença dos fótons $a$ e $b$). Os osciladores começam em $|n\rangle$. O estado ideal é **puro**:
$$|\psi_{\rm id}\rangle=c_{00}|00\rangle+c_{01}|0\tilde\xi_b\rangle+c_{10}|\tilde\xi_a0\rangle+e^{i\phi}c_{11}|\tilde\xi_a\tilde\xi_b\rangle .$$

O estado real dos fótons, com os osciladores traçados, é **misto**:
$$\rho=\sum_m|{\rm out}_m\rangle\langle{\rm out}_m|,\qquad |{\rm out}_m\rangle=\langle m|U|\psi,n\rangle .$$

Para um estado puro e outro misto, a fidelidade ([Wikipedia](https://en.wikipedia.org/wiki/Fidelity_of_quantum_states)) é
$$F(\psi_{\rm id},\rho)=\langle\psi_{\rm id}|\rho|\psi_{\rm id}\rangle=\sum_m\big|\langle\psi_{\rm id}|{\rm out}_m\rangle\big|^2=\sum_m\Big|\sum_k|c_k|^2x_k(m)\Big|^2,$$
$$x_{00}=\delta_{m_an_a}\delta_{m_bn_b},\quad x_{01}=\delta_{m_an_a}\tau_b(m_b),\quad x_{10}=\tau_a(m_a)\delta_{m_bn_b},\quad
x_{11}=e^{i\phi}\big[\tau_a(m_a)\tau_b(m_b)+\mathcal N(m)\big].$$
A soma é **sobre os módulos ao quadrado**: configurações finais diferentes dos osciladores são ortogonais e não interferem.

**Média de Haar em 2 qubits** ($d=4$): $\overline{|c_k|^2|c_l|^2}=(1+\delta_{kl})/20$, e então
$$F_1=\sum_m x(m)^\dagger W_1\,x(m),\qquad W_1=\frac{J+I}{20}\qquad(\texttt{W1}).$$
Sem osciladores, $x=(1,1,1,e^{i\phi}{\rm Ov})$ e isso é a Eq. (A2) do artigo, $F_1=[6+3\,{\rm Re}(e^{i\phi}{\rm Ov})+|{\rm Ov}|^2]/10$. A Fig. 3(a) plota $F_1$.

**Truncamento da soma em $m$ (`max_excitados`, `dq_max`).** São $2N$ osciladores, e a soma completa é exponencial em $N$. **`configuracoes`** gera a configuração elástica ($m=n$) e todas as que têm no máximo `max_excitados` osciladores com $m\ne n$, cada um com $|m-n|\le$ `dq_max`. Como $W_1$ é positiva, **cada configuração contribui com um termo $\ge0$**. Truncar dá, portanto, um **limite inferior** de $F_1$, que sobe monotonamente com `max_excitados`. Os processos que contam são:
* $m=n$: canal elástico;
* um oscilador excitado: o fóton sai com a cor deslocada de $\pm\omega$ (só contribui se $\sigma$ não for $\ll\omega$);
* dois excitados com deslocamentos opostos (por exemplo, $+\omega$ num sítio e $-\omega$ noutro): o fóton **volta à cor original** e contribui mesmo para $\sigma$ pequeno. Esse é o processo dominante para $n\ge1$. Para $n=0$ ele não existe, porque o oscilador não pode descer.

**Simetria de espelho (`espelho_simetrico`).** O fóton $a$ vai $1\to N$ e o $b$ vai $N\to1$. Se os sítios são idênticos e os osciladores $a$ e $b$ também, a configuração $(m_a,m_b)$ contribui exatamente como $(m_b$ invertida$,\ m_a$ invertida$)$. Só metade das configurações é calculada (com peso 2).

In [ ]:
for Fv in [-1, 1, 0.3 + 0.2j]:
    x = np.array([1, 1, 1, np.exp(1j * c.phi) * Fv])
    print(f'Ov={Fv}:  x^T W1 x = {np.real(x.conj() @ c.W1 @ x):.6f}   (A2) = {(6 + 3*np.real(np.exp(1j*c.phi)*Fv) + abs(Fv)**2)/10:.6f}')

cad = c.montar_cadeia(3, 1.0, 0.01, 1, 1.0, 0.01, 1)
for e in [0, 1, 2]:
    print(f'N=3, n=1, sigma=0.2, max_excitados={e}:  F1 = {c.fidelidade(cad, 0.2, max_exc=e):.6f}')

## 10. Quão bom é o truncamento? — `prob_um_foton`, `prob_um_foton_exata`

Para a parte de um fóton existe um cálculo **exato** (todas as ordens) que não enumera configurações. Cada oscilador é tocado uma única vez. Então traçar os osciladores sítio a sítio é aplicar um canal ao fóton. O estado é guardado como $R_q(\nu,\nu')$, indexado pelo deslocamento acumulado $q$, que é o mesmo no ket e no bra porque o $m$ é o mesmo:
$$R_q(\nu,\nu')\;\longrightarrow\;\sum_m t_m(\nu-q)\,t_m^*(\nu'-q)\,R_q(\nu,\nu'),\qquad q\to q+\delta_m,$$
$$P_c=\sum_m|\tau_c(m)|^2=\sum_q\langle\tilde\xi(\cdot-q)|R_q|\tilde\xi(\cdot-q)\rangle .$$
Comparar com a soma truncada (**`prob_um_foton`**) mede quanto do peso inelástico que volta ao fóton ideal o truncamento captura. O mesmo truncamento é usado no termo de dois fótons.

In [ ]:
s = 0.05
print(' n   N   P_elastico   P_truncado(2)   P_exato')
for n0 in [0, 1, 2]:
    for N in [5, 10, 20]:
        cad = c.montar_cadeia(N, 1.0, 0.01, n0, 1.0, 0.01, n0)
        print(f'{n0:2d} {N:3d}   {c.prob_um_foton(cad, "a", s, 0):.4f}       {c.prob_um_foton(cad, "a", s, 2):.4f}'
              f'          {c.prob_um_foton_exata(cad, "a", s):.4f}')

Com $S=0.01$ e $\omega=\gamma$:
* $n=0$: o truncamento é exato. Não há processo de ida e volta, e os deslocados não se sobrepõem ao fóton ideal para $\sigma\ll\omega$.
* $n=1$: `max_excitados = 2` captura quase todo o peso (em $N=20$, $0.202$ de $0.207$).
* $n=2$, $N=20$: faltam processos com 4 ou mais osciladores excitados. A curva é um **limite inferior** (a parte de um fóton captura $0.093$ de $0.108$).

## 11. Grade e parâmetros numéricos — `grade`, `pontos`

* Janela: $|\omega-\omega_c|\le k_\sigma\sigma$, com `k_sigma` = 9.
* Passo $h$: resolve $\min(\sigma,\gamma/2N)$ com `pts_per_width` pontos. O produto de $N$ fatores $t$ gira $\sim4N/\gamma$ rad por unidade de frequência perto da ressonância. A convergência é espectral: com 6 pontos já se reproduz a Fig. 3(a) a $10^{-6}$.
* `n_min` $\le n\le$ `n_max` (sempre ímpar).
* As convoluções têm comprimento $2n-1$ (FFT de tamanho `next_fast_len(2n-1)`, sem aliasing).

## 12. Validações — `validar()`

`python3 calcular_fock_N.py` roda primeiro as validações:
1. **$S=0$** reproduz a Fig. 3(a) de Brod–Combes (`infidelidade/data/fig3_L0_...npz`) para $N=1,3,10$;
2. **$N=1$** com a soma completa em $m$ reproduz o código de um sítio (`2foton-1sitio-fock/calcular_fock.py`) para $n=0,1,2$;
3. **$\mathcal N(m)$ por FFT = integral 3D direta** da Eq. (N11), com deslocamentos e sítios diferentes;
4. **conservação da probabilidade** de um fóton em dois sítios.

A célula abaixo roda todas (cerca de 30 s).

In [ ]:
c.validar()

## 13. Varredura, saída e figura — `calcular_caso`, `main`, `plot_fock_N.py`

**`calcular_caso(N, w_a, S_a, n_a, w_b, S_b, n_b)`** monta a cadeia e calcula $F_1$ em `n_sigma` valores de $\sigma$ (log, de `sigma_min` a `sigma_max`), em paralelo (`n_proc` processos). Ele salva em `data/` o `.npz` `fid_N{N}_chi.._gam.._w0.._wa.._Sa.._na.._wb.._Sb.._nb.._exc{max_excitados}_{referencia}.npz`, com `sigmas`, `F1` e todos os parâmetros. **`main()`** roda as validações e todos os `casos` $\times$ `N_values`.

**`plot_fock_N.py`** desenha uma figura por caso de `casos_plot`, no estilo da Fig. 3(a):
* cores por $N$: a paleta padrão do Mathematica, as mesmas do artigo e de `infidelidade/plot_infidelidade.py` (`MMA`);
* só as curvas com osciladores. A Fig. 3(a) sem osciladores está em `infidelidade/`, e também é gerada aqui com o caso `S = 0`;
* rótulos "N = k" em caixas cinzas à esquerda da subida de cada curva. Um rótulo que colidiria com outro desce ao longo da própria curva;
* se faltar o `.npz` de algum $N$, o script avisa e desenha os demais.

Ele salva `fig3a_fock_wa{w}_S{S}_n{n}.svg` e `.png`.

Para gerar tudo:
```
python3 calcular_fock_N.py          # soma truncada em m (qualquer omega)
python3 calcular_fock_N_lenta.py    # mola lenta, omega << gamma (Secao 14)
python3 plot_fock_N.py
```

In [ ]:
from IPython.display import SVG, display
import glob, os
for f in sorted(glob.glob('fig3a_*.svg')):
    print(os.path.basename(f)); display(SVG(f))

## 14. Regime de mola lenta, $\omega^{(a,b)}\ll\gamma$ — `calcular_fock_N_lenta.py`

**O que fica fixo quando $\omega\to0$?**
* **$S$ fixo:** então $g=\sqrt S\,\omega/2\to0$. Como $\sum_kF_{km}F_{kn}=\delta_{mn}$, vale $t_{mn}\to\delta_{mn}(-\Gamma^*/\Gamma)$: o oscilador se desacopla e tudo tende a Brod–Combes. O código da Seção 9 já trata esse caso.
* **$g$ fixo** (o caso físico de uma mola mole): $S=(2g/\omega)^2\to\infty$. Cada passagem muda o oscilador de muitos quanta, e a truncagem em `max_excitados` deixa de valer. Os próprios $F_{kn}$ perdem precisão numérica para $S\gtrsim50$.

**O limite estático.** Com $\omega\to0$, a posição $X=b+b^\dagger$ comuta com o hamiltoniano. Durante a passagem dos fótons, o oscilador fica parado e só desloca a ressonância do seu átomo:
$$\Delta\;\longrightarrow\;\Delta+2gX .\qquad\text{(L1)}$$
Em $|n\rangle$, a média de $2gX$ é zero e a variância é $4g^2(2n+1)$. São a média e a variância de $(k-n)\omega-S\omega$ com os pesos $F_{kn}^2$: é a distribuição de Franck–Condon vista com $\omega\to0$. A matriz $S$ fica diagonal em $X$, e o deslocamento de cor $(m-n)\omega$ vai a zero.

**Fidelidade de estado misto sem truncar.** Com $x_k(m)=\langle m|\hat X_k|n\rangle$ e $\hat X_k$ diagonal em $X$,
$$F_1=\sum_m x(m)^\dagger W_1x(m)=\langle n|\hat x^\dagger W_1\hat x|n\rangle
=\int d^{2N}X\;\prod_{i=1}^{2N}|\psi_{n_i}(X_i)|^2\;x(X)^\dagger W_1\,x(X).\qquad\text{(L2)}$$
A soma sobre **todos** os estados finais $m$, em todas as ordens, está incluída. Para cada $X$, as amplitudes $x(X)$ são as de Brod–Combes para $N$ sítios com **uma dessintonia por átomo** (`amplitudes_bloco`):
* $t_k=-\Gamma_k^*/\Gamma_k$, com $\Gamma_k=\gamma/2+i(\Delta+2gX_k-\omega)$;
* $\tau$ e $\mathcal N_j$ são as das Seções 7 e 8, com os produtos $\prod_{k<j}$ e $\prod_{k>j}$ calculados por produtos acumulados;
* o kernel é $\Sigma=\gamma+i(\Delta^a_j+\Delta^b_j-E)$.

**Fóton ideal.** É o mesmo `'elastica'`, tomado com $\omega\to0$: a fase de $\prod_k\langle n|t_k|n\rangle$, com a média em $X$ feita por Gauss–Hermite (**`fase_ref`**, **`nos_X`**).

**Distribuição de $X$.** $|\psi_n(X)|^2=H_n(x)^2e^{-x^2}/(2^nn!\sqrt\pi)$ com $x=X/\sqrt2$ (**`densidade_X`**). Os pontos vêm de Sobol embaralhado em $2N$ dimensões, transformado pela inversa da distribuição acumulada (**`quantis_X`**, **`amostras_X`**). São `n_embaralhamentos` conjuntos independentes de $2^{\texttt{log2\_amostras}}$ pontos. O erro estatístico é o desvio entre as médias desses conjuntos. As mesmas amostras servem para todo $\sigma$, e por isso as curvas são suaves.

**Parâmetros novos** (`casos = [(g_a, n_a, g_b, n_b), ...]`):

| variável | significado |
|---|---|
| `g_a`, `g_b` | acoplamento átomo–oscilador $g$; largura do deslocamento $2g\sqrt{2n+1}$ |
| `n_a`, `n_b` | estado de Fock inicial |
| `log2_amostras`, `n_embaralhamentos`, `semente` | amostragem quase-Monte Carlo |
| `n_hermite` | nós de Gauss–Hermite para $\langle n|t|n\rangle$ |
| `mem_bloco` | tamanho dos blocos de amostras (memória) |

A frequência $\omega$ não entra no cálculo, só na validade: $\omega\ll\min(\sigma,\gamma/N)$, isto é, o oscilador quase não gira durante a passagem do pacote. As correções são $O(\omega/\sigma,\,N\omega/\gamma)$.

**Validações** (`validar()` do novo arquivo):
1. $g=0$ reproduz Brod–Combes ($N=1,5,20$);
2. $\langle n|t|n\rangle$ estático coincide com o $t_{nn}$ de Franck–Condon do código exato com $\omega$ pequeno, com diferença $\propto\omega$;
3. para $N=1$, (L2) coincide com a **soma exata sobre todos os $m$** do código da Seção 9 ($\omega=0.05$, $S=4$): diferença $\sim10^{-4}$;
4. o erro estatístico fica abaixo de $3\times10^{-3}$.

A célula abaixo mostra a convergência para o limite estático no caso de um sítio.

In [ ]:
import calcular_fock_N_lenta as L
g, n0, s = 0.05, 1, 0.3
print(f'limite estatico (omega -> 0):  F1 = {L.fidelidade(1, g, n0, g, n0, s):.5f}')
c.dq_max = 8
for w in [0.2, 0.1, 0.05]:
    S = (2 * g / w)**2
    cad = c.montar_cadeia(1, w, S, n0, w, S, n0)
    print(f'exato, omega={w:<5} (S={S:5.2f}):  F1 = {c.fidelidade(cad, s, max_exc=2):.5f}')
c.dq_max = 1

In [ ]:
X = np.linspace(-6, 6, 400)
plt.figure(figsize=(5, 3))
for n0 in [0, 1, 2]:
    plt.plot(2 * 0.05 * X, L.densidade_X(X, n0) / (2 * 0.05), label=f'n={n0}')
plt.xlabel(r'deslocamento da ressonância $2gX$  ($g=0.05$)'); plt.ylabel('densidade')
plt.legend(); plt.tight_layout(); plt.show()